In [ ]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

# Machine Learning Assignment 2
## Classification Models and Performance Evaluation

**Dataset:** Breast Cancer Wisconsin Diagnostic Dataset

**Objective:** To implement and compare five machine learning
classification algorithms using different evaluation metrics.

In [ ]:
# Importing the required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

# Dataset download utilities
from pathlib import Path
from urllib.request import urlretrieve

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("All libraries imported successfully!")

## 1. Dataset Collection and Loading

The Breast Cancer Wisconsin Diagnostic dataset is obtained
from the UCI Machine Learning Repository.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

data_folder = Path("data")
data_folder.mkdir(exist_ok=True)

dataset_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"

dataset_path = data_folder / "wdbc.data"

if not dataset_path.exists():
    urlretrieve(dataset_url, dataset_path)
    print("Dataset downloaded successfully!")
else:
    print("Dataset already exists.")

print("Dataset path:", dataset_path)

In [ ]:
base_features = [
    "radius", "texture", "perimeter", "area", "smoothness",
    "compactness", "concavity", "concave_points", "symmetry",
    "fractal_dimension"
]

feature_names = [
    f"{feature}_{suffix}"
    for suffix in ["mean", "se", "worst"]
    for feature in base_features
]

column_names = ["id", "diagnosis"] + feature_names

df = pd.read_csv(
    dataset_path,
    header=None,
    names=column_names
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

## 2. Exploratory Data Analysis

The dataset is examined to understand its structure,
feature distributions, missing values and target classes.

In [ ]:
print("Dataset dimensions:", df.shape)

print("\nColumn information:")
df.info()

In [ ]:
df.describe()

In [ ]:
print("Total missing values:", df.isnull().sum().sum())

df.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
print("Duplicate records:", df.duplicated().sum())

In [ ]:
print(df["diagnosis"].value_counts())

print("\nPercentage distribution:")
print(df["diagnosis"].value_counts(normalize=True) * 100)

## 3. Data Visualisation

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(data=df, x="diagnosis")

plt.title("Benign vs Malignant Cases")
plt.xlabel("Diagnosis")
plt.ylabel("Count")

plt.show()

In [ ]:
plt.figure(figsize=(18, 12))

correlation_matrix = df[feature_names].corr()

sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0
)

plt.title("Feature Correlation Heatmap")
plt.show()

## 4. Data Preprocessing

The identifier column is removed because it does not
provide meaningful information for classification.

The diagnosis column is converted into numerical labels:
0 represents Benign and 1 represents Malignant.

In [ ]:
# Reload the original dataset

df = pd.read_csv(
    dataset_path,
    header=None,
    names=column_names
)

print("Original dataset restored.")
print("Dataset shape:", df.shape)

df.head()

In [ ]:
# Remove the identifier column

df = df.drop(columns=["id"])

# Convert diagnosis into numerical values

df["diagnosis"] = df["diagnosis"].map({
    "B": 0,
    "M": 1
})

print("Target encoding completed.")

print(df["diagnosis"].value_counts())

print("\nFinal dataset shape:", df.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

## 5. Model Training and Performance Evaluation

Five classification algorithms are implemented using the
training dataset. Their performance is evaluated using
accuracy, AUC, precision, recall, F1 score and MCC.

In [ ]:

# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

import joblib
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

# ============================================================
# 2. CREATE MODEL DIRECTORY
# ============================================================

model_folder = Path("model")
model_folder.mkdir(exist_ok=True)

# ============================================================
# 3. INITIALIZE FIVE CLASSIFICATION MODELS
# ============================================================

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(
            n_neighbors=5
        ))
    ]),

    "Naive Bayes": GaussianNB(),

    "Support Vector Machine": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            probability=True,
            random_state=42
        ))
    ])
}

# ============================================================
# 4. TRAIN MODELS AND CALCULATE EVALUATION METRICS
# ============================================================

results = []

model_filenames = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "k_nearest_neighbors.joblib",
    "Naive Bayes": "gaussian_naive_bayes.joblib",
    "Support Vector Machine": "support_vector_machine.joblib"
}

for model_name, model in models.items():

    print("\n" + "=" * 60)
    print("MODEL:", model_name)
    print("=" * 60)

    # Train the model
    model.fit(X_train, y_train)

    # Generate predictions
    y_pred = model.predict(X_test)

    # Get probability of positive class (Malignant = 1)
    y_probability = model.predict_proba(X_test)[:, 1]

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)

    auc = roc_auc_score(y_test, y_probability)

    precision = precision_score(
        y_test, y_pred, zero_division=0
    )

    recall = recall_score(
        y_test, y_pred, zero_division=0
    )

    f1 = f1_score(
        y_test, y_pred, zero_division=0
    )

    mcc = matthews_corrcoef(y_test, y_pred)

    # Store the results
    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "AUC": auc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "MCC": mcc
    })

    # Display confusion matrix
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

    # Display classification report
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=["Benign", "Malignant"],
        zero_division=0
    ))

    # Save trained model
    joblib.dump(
        model,
        model_folder / model_filenames[model_name]
    )

    print("Model saved successfully!")

# ============================================================
# 5. CREATE MODEL COMPARISON TABLE
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.set_index("Model")

print("\n\nFINAL MODEL COMPARISON")
print("=" * 90)

display(results_df.round(4))

# ============================================================
# 6. IDENTIFY THE BEST MODEL BASED ON ACCURACY
# ============================================================

best_model_name = results_df["Accuracy"].idxmax()

print("\nBest model based on accuracy:", best_model_name)

# ============================================================
# 7. SAVE EVALUATION RESULTS
# ============================================================

results_df.to_csv("model_metrics.csv")

# ============================================================
# 8. SAVE TEST DATA FOR STREAMLIT APPLICATION
# ============================================================

test_data = X_test.copy()

test_data["diagnosis"] = y_test

test_data.to_csv("test_data.csv", index=False)

print("\n" + "=" * 60)
print("MODEL TRAINING AND EVALUATION COMPLETED")
print("=" * 60)

print("Saved models:", len(models))
print("Saved evaluation results: model_metrics.csv")
print("Saved test dataset: test_data.csv")


## 6. Comparison of Classification Models

In [ ]:

# Compare the performance of all five models

metrics_to_plot = [
    "Accuracy", "AUC", "Precision",
    "Recall", "F1 Score", "MCC"
]

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

for ax, metric in zip(axes.flatten(), metrics_to_plot):

    results_df[metric].sort_values().plot(
        kind="bar",
        ax=ax,
        title=metric
    )

    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=35)

plt.suptitle("Performance Comparison of Five Classification Models")
plt.tight_layout()
plt.show()


In [ ]:
print("Project files generated:")

for file in Path(".").iterdir():
    print(file.name)

print("\nSaved models:")

for file in model_folder.glob("*.joblib"):
    print(file.name)